# 02. 모델 비교 — Batch 1 (그룹 교차검증)

- 규칙(`NOTES.md` 섹션 4): **Batch 1만 사용**. Batch 2는 모델·하이퍼파라미터를 확정한 뒤 최종 평가 1회 전용이라 이 노트북에서는 읽지 않음. Batch 3는 EDA 전용.
- 타깃 `cycle_life`(회귀), 주 지표 MAPE, 보조 MAE·RMSE·R². `random_state=42`.
- 입력: **A = `dQ_log_var` 1개**(기본). **B = A + `Tavg_mean`, `chargetime_mean`, `c_avg`**(Ridge·Elastic Net·트리가 보조 feature로 이득을 보는지 비교용, 기본 입력이 아님).
- 검증: 정책 23종 중 5종을 hold-out으로 떼고, 나머지에서 **정책 단위 5-fold CV를 반복**(같은 정책 셀은 항상 같은 fold). 하이퍼파라미터는 각 train fold 안에서 정책 단위 3-fold로 고르는 **중첩 CV**.
- 후보 모델: Median Baseline / Linear / Ridge / Elastic Net / Random Forest / Gradient Boosting.
- 최종 모델 선정 기준(README): ① CV 평균 MAPE ② fold별 표준편차 ③ hold-out 지표 ④ Train–Valid gap ⑤ 계수 안정성 ⑥ 해석 가능성.

In [1]:
import numpy as np, pandas as pd, re, warnings
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')
SEED, K, N_REP = 42, 5, 5

d = pd.read_csv('../data/cache/batch1_summary.csv'); z = np.load('../data/cache/batch1_dq.npz')
cells = d.drop_duplicates('cell')[['cell', 'cycle_life', 'policy']].reset_index(drop=True)
cells['dQ_log_var'] = np.log10(np.abs(np.nanvar(z['q100'] - z['q10'], axis=1)))
e = d[d.cycle <= 100].groupby('cell')
cells = cells.join(pd.DataFrame({'Tavg_mean': e.Tavg.mean(), 'chargetime_mean': e.chargetime.mean()}), on='cell')
def c_avg(p):
    a, soc, b = map(float, re.match(r'([\d.]+)C\((\d+)%\)-([\d.]+)C', p).groups()); return a*soc/100 + b*(1-soc/100)
cells['c_avg'] = cells.policy.map(c_avg)
A = ['dQ_log_var']; B = A + ['Tavg_mean', 'chargetime_mean', 'c_avg']
y, groups = cells.cycle_life.values, cells.policy.values

tr_all, ho = next(GroupShuffleSplit(1, test_size=5, random_state=SEED).split(cells, groups=groups))   # 정책 5종을 hold-out
print(f'train {len(tr_all)}셀({len(set(groups[tr_all]))}정책) / hold-out {len(ho)}셀({len(set(groups[ho]))}정책)')
print(f'수명 범위: train {y[tr_all].min():.0f}~{y[tr_all].max():.0f}(중앙값 {np.median(y[tr_all]):.0f}) / hold-out {y[ho].min():.0f}~{y[ho].max():.0f}(중앙값 {np.median(y[ho]):.0f})')
print('hold-out 정책:', sorted(set(groups[ho])))

train 35셀(18정책) / hold-out 11셀(5정책)
수명 범위: train 534~1227(중앙값 857) / hold-out 691~1190(중앙값 860)
hold-out 정책: ['3.6C(80%)-3.6C', '5.4C(60%)-3C', '5.4C(70%)-3C', '6C(50%)-3C', '7C(30%)-3.6C']


**설명 (데이터 준비)**
- Batch 1의 셀 단위 요약표(`batch1_summary.csv`)와 사이클별 Q(V) 곡선(`batch1_dq.npz`)을 읽어 **셀 1개 = 행 1개**인 표 `cells`를 만든다 (46셀).
- 핵심 feature `dQ_log_var` = 사이클 100과 10의 방전 용량 곡선 차이(ΔQ)의 분산에 log10을 취한 값. 논문의 variance feature와 같은 아이디어.
- 입력 두 가지: **A = `dQ_log_var` 1개**(기본), **B = A + 온도·충전시간·평균 C-rate**(보조 feature가 도움이 되는지 보는 용도).
- 검증용 분할: 충전 정책 23종 중 5종(11셀)을 통째로 hold-out으로 떼고, 나머지 35셀(18정책)로 CV. 같은 정책 셀이 train/valid에 섞이지 않게 **정책 단위**로 나눔.

In [2]:
MAPE_SCORE = 'neg_mean_absolute_percentage_error'
def _div(v): return v / 1000
def _mul(v): return v * 1000
def wrap(est, log=False):   # 타깃을 1000사이클 단위(또는 log10)로 바꿔 학습 → 규제 크기가 수명 단위에 좌우되지 않게
    return TransformedTargetRegressor(est, func=np.log10 if log else _div, inverse_func=(lambda v: 10 ** v) if log else _mul, check_inverse=False)

def build(name, log=False):
    """(추정기, 하이퍼파라미터 격자) — 격자 키는 TransformedTargetRegressor 안쪽 기준."""
    if name == 'Median Baseline': return DummyRegressor(strategy='median'), None
    if name == 'Linear':          return wrap(make_pipeline(StandardScaler(), LinearRegression()), log), None
    if name == 'Ridge':           return wrap(make_pipeline(StandardScaler(), Ridge()), log), {'regressor__ridge__alpha': [.01, .1, 1, 10, 100]}
    if name == 'Elastic Net':     return wrap(make_pipeline(StandardScaler(), ElasticNet(max_iter=20000)), log), {'regressor__elasticnet__alpha': [.001, .01, .1, 1], 'regressor__elasticnet__l1_ratio': [.2, .5, .8, 1.]}
    if name == 'Random Forest':   return wrap(RandomForestRegressor(n_estimators=200, random_state=SEED), log), {'regressor__max_depth': [2, 3, None], 'regressor__min_samples_leaf': [1, 3]}
    if name == 'Gradient Boosting': return wrap(GradientBoostingRegressor(random_state=SEED), log), {'regressor__n_estimators': [50, 100], 'regressor__learning_rate': [.05, .1], 'regressor__max_depth': [1, 2]}

class Avg:   # 학습된 모델들의 예측을 단순 평균 (가중치를 학습하지 않음 → 46셀에서 과적합 방지)
    def __init__(self, *ms): self.ms = ms; self.best_params_ = {}
    def predict(self, X): return np.mean([m.predict(X) for m in self.ms], axis=0)

def fit_one(name, cols, idx_tr, log=False, inner=3):
    if name == 'Linear+GB 평균': return Avg(*[fit_one(n, cols, idx_tr, log, inner) for n in ('Linear', 'Gradient Boosting')])
    est, grid = build(name, log); X, g = cells.loc[idx_tr, cols].values, groups[idx_tr]
    if grid: est = GridSearchCV(est, grid, scoring=MAPE_SCORE, cv=GroupKFold(inner), n_jobs=-1)
    return est.fit(X, y[idx_tr], groups=g) if grid else est.fit(X, y[idx_tr])

def best_params(m): return getattr(m, 'best_params_', {})
def coefs(m):
    r = getattr(m, 'best_estimator_', m)
    try: return r.regressor_[-1].coef_
    except Exception: return None

def metrics(yt, p): return {'MAPE': np.mean(np.abs(yt - p) / yt), 'MAE': mean_absolute_error(yt, p), 'RMSE': mean_squared_error(yt, p) ** .5, 'R2': r2_score(yt, p)}

def group_folds(idx, rep):
    g = groups[idx]; rng = np.random.RandomState(SEED + rep); pols = rng.permutation(np.unique(g)); fo = {p: i % K for i, p in enumerate(pols)}
    f = np.array([fo[x] for x in g]); return [(idx[f != j], idx[f == j]) for j in range(K)]

def cv_run(name, cols, idx=tr_all, log=False, n_rep=N_REP):
    """정책 단위 K-fold를 n_rep회 반복(중첩 튜닝). 반복별 pooled out-of-fold 지표, fold별 MAPE, train MAPE, 계수, OOF 예측을 모음."""
    per_rep, fold_mape, train_mape, co, oof = [], [], [], [], np.full((n_rep, len(y)), np.nan)
    for r in range(n_rep):
        for tr, va in group_folds(idx, r):
            m = fit_one(name, cols, tr, log); p = m.predict(cells.loc[va, cols].values); oof[r, va] = p
            fold_mape.append(np.mean(np.abs(y[va] - p) / y[va])); train_mape.append(np.mean(np.abs(y[tr] - m.predict(cells.loc[tr, cols].values)) / y[tr]))
            c = coefs(m); co.append(c) if c is not None else None
        per_rep.append(metrics(y[idx], oof[r, idx]))
    return {'rep': pd.DataFrame(per_rep), 'fold_mape': np.array(fold_mape), 'train_mape': np.mean(train_mape), 'coef': np.array(co) if co else None, 'oof': oof}

**설명 (공통 함수)**
- `fit_one`: 모델 하나를 학습. Ridge·Elastic Net·트리는 학습 데이터 안에서 한 번 더 정책 단위 CV로 하이퍼파라미터를 고름(중첩 CV → 고른 값이 검증 데이터를 보지 않음).
- `wrap`: 타깃을 1000사이클 단위로 나눠 학습하고 예측 때 되돌림(규제 크기가 수명 단위에 좌우되지 않게).
- `cv_run`: 정책 단위 5-fold CV를 5번 반복(분할 시드만 바꿈). 반복별 지표, fold별 MAPE, train MAPE, 계수, out-of-fold 예측을 모아 둠.
- `metrics`: MAPE(주 지표) · MAE · RMSE · R².
- 이후 셀들은 전부 이 함수들을 재사용한다.

## 1. 모델 비교 — 정책 단위 그룹 CV (train 18정책)

모델 × 입력(A: `dQ_log_var` 1개, B: A + 보조 3개)을 같은 분할로 비교. `MAPE`는 반복 평균, `fold 표준편차`는 모든 fold의 MAPE 표준편차, `gap`은 valid MAPE − train MAPE.

In [3]:
runs, rows = {}, []
plan = [('Median Baseline', ['A']), ('Linear', ['A', 'B']), ('Ridge', ['A', 'B']), ('Elastic Net', ['A', 'B']), ('Random Forest', ['A', 'B']), ('Gradient Boosting', ['A', 'B'])]
for name, sets in plan:
    for s in sets:
        cols = A if s == 'A' else B; r = cv_run(name, cols); runs[(name, s)] = r; m = r['rep'].mean()
        rows.append({'모델': name, '입력': s, 'CV MAPE': m.MAPE, 'MAE': m.MAE, 'RMSE': m.RMSE, 'R2': m.R2, 'fold 표준편차': r['fold_mape'].std(), 'train MAPE': r['train_mape'], 'gap': m.MAPE - r['train_mape']})
cv_table = pd.DataFrame(rows).set_index(['모델', '입력']); cv_table.style.format({'CV MAPE': '{:.1%}', 'MAE': '{:.0f}', 'RMSE': '{:.0f}', 'R2': '{:.2f}', 'fold 표준편차': '{:.1%}', 'train MAPE': '{:.1%}', 'gap': '{:+.1%}'})

**설명 (모델 비교 CV)**
- 6개 모델(중앙값 기준선, Linear, Ridge, Elastic Net, Random Forest, Gradient Boosting) × 입력 A/B를 같은 분할로 비교.
- `CV MAPE`는 5회 반복 평균, `gap`은 valid MAPE − train MAPE(클수록 과적합).
- 읽는 법: 입력 A에서 선형 계열이 7.3%로 같고 gap이 가장 작음 → feature가 1개라 규제(Ridge 등)가 의미 없고 단순 선형이면 충분.

**결과 · 해석 (그룹 CV)**
- 입력 A에서 **Linear·Ridge·Elastic Net이 모두 7.3%로 동일**(feature 1개라 규제 효과 없음 → 가장 단순한 Linear로 충분). fold 표준편차 1.8%, gap +0.5%p로 가장 낮고 안정적. 중앙값 기준선(21.0%)의 약 1/3.
- 트리(RF 9.2%, GB 8.7%)는 CV에서 선형보다 **나쁘고** gap이 큼(+4.0~4.5%p, train MAPE 4~5% vs valid 9%) → 과적합 경향.
- 입력 B(보조 feature 3개 추가)는 선형 계열에서 오히려 나빠지고(7.3% → 7.6~7.7%) fold 표준편차도 커짐(1.8% → 2.6~2.9%) → **H1 재확인**. 트리는 B에서도 큰 차이 없음.

## 2. Hold-out (정책 5종) — 각 모델을 train 전체로 다시 학습해 한 번만 평가

In [4]:
rows = []
for name, sets in plan:
    for s in sets:
        cols = A if s == 'A' else B; m = fit_one(name, cols, tr_all, inner=5); p = m.predict(cells.loc[ho, cols].values)
        rows.append({'모델': name, '입력': s, **metrics(y[ho], p), '선택된 하이퍼파라미터': str({k.split('__')[-1]: v for k, v in best_params(m).items()})})
ho_table = pd.DataFrame(rows).set_index(['모델', '입력']); ho_table.style.format({'MAPE': '{:.1%}', 'MAE': '{:.0f}', 'RMSE': '{:.0f}', 'R2': '{:.2f}'})

**설명 (hold-out 평가)**
- CV에 쓰지 않은 정책 5종(11셀)에 각 모델을 한 번 평가. 모델은 35셀 전체로 다시 학습.
- 읽는 법: 여기서는 GB·RF가 선형보다 좋게 나오지만(CV와 반대), 아래 셀에서 보듯 3셀이 만든 결과라 단독으로 믿으면 안 됨.

In [5]:
# hold-out 오차 분해: 어느 정책에서 선형이 크게 틀리는가 (오차율 = (예측−실제)/실제)
ml, mg = fit_one('Linear', A, tr_all, inner=5), fit_one('Gradient Boosting', A, tr_all, inner=5)
dd = cells.loc[ho, ['policy', 'cycle_life', 'dQ_log_var']].copy(); dd['선형 예측'] = ml.predict(cells.loc[ho, A].values).round(0); dd['GB 예측'] = mg.predict(cells.loc[ho, A].values).round(0)
dd['선형 오차율'] = (dd['선형 예측'] - dd.cycle_life) / dd.cycle_life; dd['GB 오차율'] = (dd['GB 예측'] - dd.cycle_life) / dd.cycle_life
display(dd.sort_values('cycle_life').style.format({'dQ_log_var': '{:.2f}', '선형 오차율': '{:+.1%}', 'GB 오차율': '{:+.1%}'}))
print('정책별 평균 |오차율|:'); display(dd.assign(선형=dd['선형 오차율'].abs(), GB=dd['GB 오차율'].abs()).groupby('policy')[['선형', 'GB']].mean().style.format('{:.1%}'))

,policy,cycle_life,dQ_log_var,선형 예측,GB 예측,선형 오차율,GB 오차율
18,5.4C(70%)-3C,691.000000,-3.65,714.000000,695.000000,+3.3%,+0.6%
35,7C(30%)-3.6C,703.000000,-3.77,781.000000,695.000000,+11.1%,-1.1%
15,5.4C(60%)-3C,719.000000,-4.02,916.000000,900.000000,+27.4%,+25.2%
34,7C(30%)-3.6C,742.000000,-3.85,823.000000,865.000000,+10.9%,+16.6%
19,5.4C(70%)-3C,788.000000,-3.62,697.000000,695.000000,-11.5%,-11.8%
28,6C(50%)-3C,860.000000,-3.84,821.000000,865.000000,-4.5%,+0.6%
14,5.4C(60%)-3C,880.000000,-4.05,932.000000,952.000000,+5.9%,+8.2%
29,6C(50%)-3C,917.000000,-3.84,817.000000,865.000000,-10.9%,-5.7%
2,3.6C(80%)-3.6C,1177.000000,-4.74,1313.000000,1199.000000,+11.6%,+1.9%
1,3.6C(80%)-3.6C,1179.000000,-5.01,1465.000000,1199.000000,+24.3%,+1.7%


정책별 평균 |오차율|:


,선형,GB
policy,,
3.6C(80%)-3.6C,19.7%,1.4%
5.4C(60%)-3C,16.7%,16.7%
5.4C(70%)-3C,7.4%,6.2%
6C(50%)-3C,7.7%,3.1%
7C(30%)-3.6C,11.0%,8.9%


**설명 (hold-out 오차 분해)**
- 셀·정책별로 선형/GB 예측과 오차율((예측−실제)/실제)을 보여줌.
- 읽는 법: 선형의 큰 오차가 `3.6C(80%)-3.6C` 3셀에 몰려 있음. 이 3셀은 `dQ_log_var`가 학습 범위(−4.44 이하)를 벗어난 극단값이라 선형이 직선으로 외삽하며 틀린 것.

**결과 · 해석 (hold-out)** — CV와 순위가 달라짐
- hold-out(11셀)에서는 GB(A) 6.7%, RF(A) 7.5%가 선형(A) 13.1%보다 크게 좋음. CV에서는 반대(선형 7.3% < GB 8.7%).
- 오차 분해 결과 선형의 오차는 `3.6C(80%)-3.6C` 3셀(수명 약 1,180, `dQ_log_var` −4.7~−5.0)에 집중됨(+12~+24% 과대예측, 정책 평균 19.7%). 이 정책이 hold-out에 있어서 train의 `dQ_log_var` 최솟값(−4.44)보다 극단값을 선형이 외삽한 것이고, GB는 이 셀들을 1,199로 예측(상한에서 포화)해 오차가 1.4%. 나머지 정책에서는 선형·GB 차이가 크지 않음(5.4C(60%)-3C는 둘 다 16.7%).
- → **hold-out 한 번은 3셀에 좌우되므로 불안정.** CV(반복 5회, 35셀)를 주 근거로 삼되, 선형이 `dQ_log_var` 극단에서 과대예측하는 약점이 있음을 기록.

## 3. 선형 계열 계수 안정성 (입력 B, CV fold마다 학습한 계수의 평균·표준편차)

표준화된 feature 기준. 부호가 fold마다 바뀌면 불안정.

In [6]:
names = ['dQ_log_var', 'Tavg_mean', 'chargetime_mean', 'c_avg']; out = {}
for name in ['Linear', 'Ridge', 'Elastic Net']:
    c = runs[(name, 'B')]['coef'] * 1000      # 타깃 1000사이클 단위로 학습했으므로 사이클 단위로 환산
    out[name + ' 평균'] = c.mean(0); out[name + ' 표준편차'] = c.std(0); out[name + ' 부호 일관(%)'] = (np.sign(c) == np.sign(c.mean(0))).mean(0) * 100
pd.DataFrame(out, index=names).round(1)

,Linear 평균,Linear 표준편차,Linear 부호 일관(%),Ridge 평균,Ridge 표준편차,Ridge 부호 일관(%),Elastic Net 평균,Elastic Net 표준편차,Elastic Net 부호 일관(%)
dQ_log_var,-105.4,24.9,100.0,-93.5,22.3,100.0,-106.0,24.0,100.0
Tavg_mean,-25.8,7.1,100.0,-27.2,6.5,100.0,-19.8,8.7,100.0
chargetime_mean,9.1,7.9,88.0,11.9,6.7,96.0,8.5,5.7,80.0
c_avg,-43.7,20.7,96.0,-48.0,14.5,100.0,-37.1,17.3,100.0


**설명 (계수 안정성)**
- 입력 B로 CV fold마다 선형 계열을 학습하고, 계수가 fold마다 같은 부호인지 확인.
- `부호 일관(%)`이 100%에 가까우면 어느 fold에서도 같은 방향으로 작용한다는 뜻. 단, Batch 1 안에서의 안정성일 뿐 다른 배치에서도 그렇다는 보장은 아님.

**결과 · 해석 (계수 안정성)**
- Batch 1 CV fold마다 학습한 계수의 부호는 대부분 일관: `dQ_log_var` −105(사이클/표준편차, 100%), `Tavg_mean` −26(100%), `c_avg` −44(96~100%), `chargetime_mean` +9(80~96%로 가장 불안정). 규제(Ridge·Elastic Net)해도 비슷.
- 주의: 이것은 **Batch 1 안의 안정성**일 뿐임. EDA에서 `Tavg`는 Batch 2에서 부호가 반대(+0.42), `chargetime`은 +0.09로 사라졌으므로 Batch 1에서 안정적이라고 일반화되는 것은 아님.

## 4. 가설 H3 — 타깃 log10 변환이 짧은 수명 구간의 MAPE를 개선하는가

선형(입력 A)에서 타깃을 원래 단위 vs `log10`으로 학습해 비교. 수명 3분위(짧음/중간/김)별 MAPE는 첫 반복의 out-of-fold 예측으로 계산.

In [7]:
h3 = {'원래 단위': runs[('Linear', 'A')], 'log10 타깃': cv_run('Linear', A, log=True)}
tert = pd.qcut(y[tr_all], 3, labels=['짧음', '중간', '김']); rows = []
for k, r in h3.items():
    p = r['oof'][0, tr_all]; ape = np.abs(y[tr_all] - p) / y[tr_all]
    rows.append({'타깃': k, '전체 MAPE': r['rep'].MAPE.mean(), **{f'{t} 구간 MAPE': ape[tert == t].mean() for t in ['짧음', '중간', '김']}, '평균 부호 오차(예측−실제)/실제': np.mean((p - y[tr_all]) / y[tr_all])})
pd.DataFrame(rows).set_index('타깃').style.format('{:.1%}')

,전체 MAPE,짧음 구간 MAPE,중간 구간 MAPE,김 구간 MAPE,평균 부호 오차(예측−실제)/실제
타깃,,,,,
원래 단위,7.3%,7.0%,8.0%,6.7%,1.0%
log10 타깃,7.7%,6.6%,8.4%,7.5%,0.6%


**설명 (가설 H3: log 타깃)**
- 수명을 log10으로 바꿔 학습하면 짧은 수명 구간 오차가 줄어드는지 확인(원래 단위 vs log).
- 수명을 3분위(짧음/중간/김)로 나눠 구간별 MAPE를 비교. 결과: 전체가 오히려 나빠져서(7.7% vs 7.3%) 원래 단위 유지.

**결과 · 해석 (H3)** — 지지되지 않음
- log10 타깃은 전체 MAPE가 오히려 **나쁨**(7.7% vs 7.3%). 짧은 구간은 소폭 좋아지지만(6.6% vs 7.0%) 긴 구간이 나빠짐(7.5% vs 6.7%).
- → **타깃은 원래 단위 유지.** 단 이는 Batch 1 수명 3분위 기준이고, Batch 1보다 짧은 Batch 2 구간에서의 효과는 판단할 수 없음.

## 5. 가설 H4 — 학습 범위 양 끝에서 트리가 선형보다 크게 틀리는가 (외삽)

전체 46셀에서 수명이 **가장 짧은 25%** 또는 **가장 긴 25%**를 test로 빼고 나머지로 학습. 정책이 셀 2개씩이라 꼬리를 자르면 대부분 정책 단위로 떨어짐. `부호 오차` > 0이면 과대예측.

In [8]:
q25, q75 = np.quantile(y, [.25, .75]); rows = []
for tail, mask in [(f'짧은 25% (<{q25:.0f})', y < q25), (f'긴 25% (>{q75:.0f})', y > q75)]:
    te, tr = np.where(mask)[0], np.where(~mask)[0]
    for name in ['Linear', 'Random Forest', 'Gradient Boosting']:
        m = fit_one(name, A, tr); p = m.predict(cells.loc[te, A].values)
        rows.append({'test 구간': tail, '모델': name, 'test 셀': len(te), 'MAPE': np.mean(np.abs(y[te] - p) / y[te]), '부호 오차': np.mean((p - y[te]) / y[te]), '예측 범위': f'{p.min():.0f}~{p.max():.0f}', '실제 범위': f'{y[te].min():.0f}~{y[te].max():.0f}'})
pd.DataFrame(rows).set_index(['test 구간', '모델']).style.format({'MAPE': '{:.1%}', '부호 오차': '{:+.1%}'})

**설명 (가설 H4: 외삽)**
- 수명이 가장 짧은 25% 또는 가장 긴 25%를 통째로 test로 빼고 나머지로 학습 → 학습 범위 밖을 맞히는 능력 확인.
- 결과: 트리(RF·GB)는 예측이 학습 범위 안에 갇혀 양 끝에서 선형보다 크게 틀림. 이것이 선형을 선택한 근거 중 하나이고, 나중에 Batch 2(더 짧은 수명)를 예측할 때 중요해짐.

**결과 · 해석 (H4)** — 지지
- **양 끝 모두 트리가 선형보다 크게 틀림**: 짧은 25% 선형 17.5% vs RF 22.7%·GB 23.2%, 긴 25% 선형 12.2% vs RF 21.4%·GB 22.4%. 트리의 예측 범위가 학습 범위에 갇힘(긴 구간 실제 917~1227인데 RF 예측 840~867).
- 선형도 완벽하지 않음: 짧은 구간 **+17.5% 과대예측**, 긴 구간 −9.8% 과소예측(평균 쪽으로 쏠림). Batch 2는 이보다 더 짧은 수명(392~514)이 77%라 선형도 더 크게 틀릴 수 있음.
- 한계: 꼬리를 셀 단위로 잘랐고(정책은 대부분 셀 2개씩이라 거의 정책 단위로 분리됨), 각 12셀이라 표본이 작음.

## 6. 가설 H5 — 선형 모델은 수명이 긴 셀을 과소예측하는가

선형(입력 A)의 반복 평균 out-of-fold 예측에서 잔차(실제 − 예측)를 실제 수명에 대해 확인. 잔차가 수명이 클수록 커지면(양(+)) 장수명 과소예측.

In [9]:
p = np.nanmean(runs[('Linear', 'A')]['oof'][:, tr_all], axis=0); res = y[tr_all] - p
from scipy.stats import spearmanr
slope = np.polyfit(y[tr_all], res, 1)[0]; rho = spearmanr(y[tr_all], res)[0]
print(f'잔차(실제−예측) vs 실제 수명: 기울기 {slope:+.3f}, Spearman {rho:+.2f}')
top = tr_all[np.argsort(y[tr_all])[-6:]]; print('수명 상위 6셀:'); display(pd.DataFrame({'실제': y[top], '예측': np.nanmean(runs[("Linear", "A")]["oof"][:, top], axis=0).round(0), 'policy': groups[top]}).assign(오차율=lambda x: (x.예측 - x.실제) / x.실제).style.format({'오차율': '{:+.1%}'}))

잔차(실제−예측) vs 실제 수명: 기울기 +0.176, Spearman +0.40
수명 상위 6셀:


,실제,예측,policy,오차율
0,1017.000000,871.000000,6C(40%)-3C,-14.4%
1,1051.000000,939.000000,8C(15%)-3.6C,-10.7%
2,1054.000000,931.000000,5.4C(40%)-3.6C,-11.7%
3,1074.000000,1001.000000,4.4C(80%)-4.4C,-6.8%
4,1226.000000,1153.000000,4C(80%)-4C,-6.0%
5,1227.000000,1267.000000,4C(80%)-4C,+3.3%


**설명 (가설 H5: 장수명 과소예측)**
- 잔차(실제−예측)가 실제 수명이 클수록 커지는지(= 장수명을 낮게 예측하는지) 상관으로 확인.
- 결과: 중간 정도로 지지(Spearman +0.40). 평균 쪽으로 쏠리는 선형의 일반적인 성질.

**결과 · 해석 (H5)** — 지지(중간 정도)
- 잔차(실제−예측)와 실제 수명의 상관이 양(+)(기울기 +0.18, Spearman +0.40) → **수명이 길수록 과소예측**하는 경향. 수명 상위 6셀 중 5셀이 과소예측(−6~−14%), 1셀만 +3%.
- 다만 hold-out에서는 `3.6C(80%)-3.6C`(수명 약 1,180)가 반대로 +12~+24% **과대예측**됨 → 극단 장수명에서 방향이 일정하지 않음. 셀 수가 적어 확정은 어렵고, 장수명 쪽 dQ 포화가 원인 후보(EDA Q3).

## 요약 · 잠정 모델 선택 (Batch 2는 아직 읽지 않음)

| 기준 | Linear (A) | Gradient Boosting (A) | Random Forest (A) |
|---|---|---|---|
| ① CV 평균 MAPE | **7.3%** | 8.7% | 9.2% |
| ② fold 표준편차 | 1.8% | 2.1% | 2.0% |
| ③ hold-out MAPE (11셀, 3셀이 한 정책) | 13.1% | **6.7%** | 7.5% |
| ④ Train–Valid gap | **+0.5%p** | +4.5%p | +4.0%p |
| ⑤ 계수 안정성 | 부호 100% 일관 | (해당 없음) | (해당 없음) |
| ⑥ 해석 가능성 | 높음 | 낮음 | 낮음 |
| 외삽 H4 (짧은25% / 긴25% MAPE) | **17.5% / 12.2%** | 23.2% / 22.4% | 22.7% / 21.4% |

- **잠정 선택: 입력 A(`dQ_log_var` 1개) + 단순 선형 회귀.** Ridge·Elastic Net은 feature가 1개라 선형과 같은 결과(7.3%)이므로 추가 이득이 없음. 보조 feature(입력 B)는 선형 계열에서 오히려 나빠짐(H1 재확인).
- **불확실한 점**: hold-out에서는 GB·RF가 선형보다 크게 좋았지만(6.7% vs 13.1%) 11셀 중 3셀(`3.6C(80%)-3.6C`)이 차이를 거의 다 만듦. CV(반복 5회, 35셀)와 외삽 검증(H4)은 선형을 지지하므로 CV를 주 근거로 판단. 최종 확정은 팀 합의 후.
- **Batch 2 예상**: Batch 2의 77%는 Batch 1 최소 수명(534)보다 짧아(392~514) 선형도 크게 과대예측할 가능성이 큼(H4에서 짧은 꼬리 +17.5%). 이는 모델 선택으로 해결되는 문제가 아니라 학습 범위 한계이므로 보고서에 한계로 명시.
- 남은 일: ① 최종 후보 확정(선형 vs GB) ② 모델·하이퍼파라미터 고정 후 Batch 2 **1회** 평가 ③ 결과를 짧은 덩어리(30셀)와 `newstructure` 9셀로 나눠 해석.

## 6. 앙상블 후보 — Linear+GB 단순 평균 (입력 A)

선형과 GB를 각각 학습해 예측을 평균. 같은 CV·hold-out·외삽(H4) 틀로 Linear, GB 단독과 비교. Batch 2는 사용하지 않음.

In [10]:
ENS = 'Linear+GB 평균'; rows = []
cv_ens = cv_run(ENS, A)
for name, r in [('Linear', runs[('Linear', 'A')]), ('Gradient Boosting', runs[('Gradient Boosting', 'A')]), (ENS, cv_ens)]:
    rows.append({'모델': name, 'CV MAPE': r['rep'].MAPE.mean(), 'fold 표준편차': r['fold_mape'].std(), 'train MAPE': r['train_mape'], 'gap': r['rep'].MAPE.mean() - r['train_mape']})
cvt = pd.DataFrame(rows).set_index('모델')
cvt['hold-out MAPE'] = [metrics(y[ho], fit_one(n, A, tr_all, inner=5).predict(cells.loc[ho, A].values))['MAPE'] for n in cvt.index]
for tail, mask in [('짧은25%', y < q25), ('긴25%', y > q75)]:
    te, tr = np.where(mask)[0], np.where(~mask)[0]
    cvt[f'H4 {tail} MAPE'] = [np.mean(np.abs(y[te] - fit_one(n, A, tr).predict(cells.loc[te, A].values)) / y[te]) for n in cvt.index]
cvt.style.format('{:.1%}').format({'gap': '{:+.1%}'})

,CV MAPE,fold 표준편차,train MAPE,gap,hold-out MAPE,H4 짧은25% MAPE,H4 긴25% MAPE
모델,,,,,,,
Linear,0.073094,0.018009,0.068162,+0.5%,0.131439,0.175461,0.121707
Gradient Boosting,0.087188,0.021028,0.042250,+4.5%,0.067184,0.232297,0.224106
Linear+GB 평균,0.074704,0.016239,0.050909,+2.4%,0.097785,0.203879,0.160988


**설명 (앙상블 시도)**
- 선형과 GB 예측을 단순 평균해 CV·hold-out·외삽에서 비교. 가중치를 학습하지 않아 46셀에서 과적합할 여지가 적음.
- 결과: 선형(7.3%)을 넘지 못해(7.5%) 채택하지 않음. "모델을 복잡하게 해도 안 좋아진다"는 근거.

## 7. 최종 평가 — Batch 1 전체로 학습 → Batch 2 평가 (1회)

**확정 모델: 입력 A(`dQ_log_var` 1개) + 선형 회귀.** Batch 1 46셀 전체로 학습하고 Batch 2를 한 번만 예측한다. 이 결과를 보고 모델·feature를 바꾸면 Batch 2 점수는 낙관적이 되므로, 수정하더라도 이 첫 결과는 그대로 기록하고 수정본은 따로 표시한다. `cycle_life`가 없는 셀은 제외.

In [11]:
final = fit_one('Linear', A, np.arange(len(cells)))   # Batch 1 46셀 전체
d2 = pd.read_csv('../data/cache/batch2_summary.csv'); z2 = np.load('../data/cache/batch2_dq.npz')
c2 = d2.drop_duplicates('cell')[['cell', 'cycle_life', 'policy']].reset_index(drop=True)
c2['dQ_log_var'] = np.log10(np.abs(np.nanvar(z2['q100'] - z2['q10'], axis=1)))
c2 = c2[c2.cycle_life.notna()].reset_index(drop=True)
c2['예측'] = final.predict(c2[A].values); c2['오차율'] = (c2.예측 - c2.cycle_life) / c2.cycle_life
c2['구간'] = np.where(c2.policy.str.contains('newstructure'), 'newstructure(9)', np.where(c2.cycle_life < 600, '짧은 덩어리', '기타 장수명'))
rows = {'Batch 2 전체': c2, **{k: g for k, g in c2.groupby('구간')}}
out = pd.DataFrame({k: {'셀': len(g), **metrics(g.cycle_life.values, g.예측.values), '평균 부호 오차': g.오차율.mean()} for k, g in rows.items()}).T
print(f'학습 계수: 기울기 {final.regressor_[-1].coef_[0] * 1000:+.0f} 사이클/표준편차, dQ_log_var 학습 범위 {cells.dQ_log_var.min():.2f}~{cells.dQ_log_var.max():.2f} / Batch 2 {c2.dQ_log_var.min():.2f}~{c2.dQ_log_var.max():.2f} (범위 밖 {((c2.dQ_log_var < cells.dQ_log_var.min()) | (c2.dQ_log_var > cells.dQ_log_var.max())).sum()}셀)')
out.style.format({'셀': '{:.0f}', 'MAPE': '{:.1%}', 'MAE': '{:.0f}', 'RMSE': '{:.0f}', 'R2': '{:.2f}', '평균 부호 오차': '{:+.1%}'})

학습 계수: 기울기 -162 사이클/표준편차, dQ_log_var 학습 범위 -5.01~-3.35 / Batch 2 -4.45~-3.09 (범위 밖 11셀)


,셀,MAPE,MAE,RMSE,R2,평균 부호 오차
Batch 2 전체,39,31.5%,156,170,0.40,+29.9%
newstructure(9),9,14.2%,125,146,-0.02,+7.2%
짧은 덩어리,30,36.7%,165,176,-25.50,+36.7%


**설명 (최종 평가: Batch 2)**
- 확정 모델(입력 A + 선형)을 Batch 1 전체 46셀로 학습하고, Batch 2의 `cycle_life`가 있는 39셀을 **딱 한 번** 예측. 이 값을 보고 모델을 바꾸지 않았음(바꾸면 점수가 낙관적).
- 구간 분해: 짧은 덩어리(30셀, 수명 392~514) / `newstructure`(9셀) 별로 오차를 따로 봄.
- 읽는 법: 전체 MAPE 31.5%는 거의 전부 짧은 덩어리의 과대예측(+36.7%)에서 나옴. 같은 `dQ_log_var`에서 Batch 2 짧은 셀이 150~200 사이클 더 짧기 때문.

## 8. 과제 양식 — Reporting format (Regression)

안내문의 회귀 보고 형식(구분 / MAPE (%) / 비고)으로 정리. 새 예측은 없고 앞 셀(CV·hold-out·7번 Batch 2 예측)의 값을 모아 계산만 함. **본 분석은 회귀 하나**이며 분류는 맨 아래 부록(논문 비교용 참고)에만 둠.

In [12]:
# 8. Reporting format (Regression) — 앞 셀의 값 재사용, Batch 2 재평가 아님
ml = fit_one('Linear', A, tr_all, inner=5)   # Valid(hold-out)용: Batch 1 train 35셀로 학습, 선형은 하이퍼파라미터 없음
p_ho = ml.predict(cells.loc[ho, A].values)
mape = lambda t, p: np.mean(np.abs(t - p) / t)
rr = runs[('Linear', 'A')]['rep']
tr_m, va_m, te_m = rr.MAPE.mean() * 100, mape(y[ho], p_ho) * 100, mape(c2.cycle_life.values, c2.예측.values) * 100
tab = pd.DataFrame({'MAPE (%)': [tr_m, va_m, te_m, va_m - tr_m, te_m - va_m, te_m - 9.1],
                    '비고': ['정책 단위 5-fold CV 5회 반복 평균(35셀)', '정책 5종 11셀, 3셀이 오차를 좌우(불안정)', 'Batch 2 39셀, 1회 평가',
                           '(+) : 과적합 의심 → 단정하지 않음(선형의 train–valid gap +0.5%p)', '(+) : 배치간 일반화 저하 의심', 'Target : 원논문 9.1%']},
                   index=['Train (Batch 1 CV)', 'Valid (Batch 1 Hold-out)', 'Test (Batch 2)', 'Gap (Train-Valid)', 'Gap (Valid-Test)', 'Gap (Target-Test)'])
tab.style.format({'MAPE (%)': '{:.1f}'})

,MAPE (%),비고
Train (Batch 1 CV),7.3,정책 단위 5-fold CV 5회 반복 평균(35셀)
Valid (Batch 1 Hold-out),13.1,"정책 5종 11셀, 3셀이 오차를 좌우(불안정)"
Test (Batch 2),31.5,"Batch 2 39셀, 1회 평가"
Gap (Train-Valid),5.8,(+) : 과적합 의심 → 단정하지 않음(선형의 train–valid gap +0.5%p)
Gap (Valid-Test),18.4,(+) : 배치간 일반화 저하 의심
Gap (Target-Test),22.4,Target : 원논문 9.1%


**설명 (Reporting format)**
- 안내문의 회귀 보고 표를 그대로 채움. Gap은 모두 (뒤 − 앞): Train-Valid = Valid − Train, Valid-Test = Test − Valid, Target-Test = Test − 논문 9.1%. 단위는 %p.
- 읽는 법: Valid-Test +18.4%p가 크게 벌어지는 구간(배치 간 일반화 저하 의심을 지지). Train-Valid +5.8%p는 hold-out 3셀(`3.6C(80%)-3.6C`)이 만든 값이라 과적합으로 단정하지 않음(선형의 train–valid gap은 +0.5%p).

## 9. 배치 간 차이 확인 — Batch 2 짧은 30셀은 Batch 1과 무엇이 다른가 (새 학습 없음)

7번의 확정 모델(`final`)과 Batch 2 예측(`c2`)을 **그대로 사용**하고 값만 비교한다. 모델·점수는 바뀌지 않음. 확인할 것: ① 짧은 30셀의 입력 값(온도·충전시간·IR·초기 용량)이 Batch 1 범위 안인가 ② `dQ_log_var`가 비슷한 Batch 1 셀과 수명을 비교하면 얼마나 차이 나는가 ③ 과대예측 크기(예측−실제)가 어떤 값과 같이 움직이는가.

In [13]:
# 셀 단위 보조 값: 초기 100사이클 **중앙값** 온도·충전시간·IR(Batch 2 chargetime에 이상치 사이클이 있어 평균은 왜곡됨), 사이클 2~10 평균 방전 용량(초기 용량)
def extra(df):
    e = df[df.cycle <= 100].groupby('cell'); f = df[(df.cycle >= 2) & (df.cycle <= 10)].groupby('cell')
    return pd.DataFrame({'Tavg': e.Tavg.median(), 'chargetime': e.chargetime.median(), 'IR': e.IR.median(), 'Q0': f.QDischarge.mean()})
x1 = cells[['cell', 'cycle_life', 'policy', 'dQ_log_var']].join(extra(d), on='cell')
x2 = c2.join(extra(d2), on='cell'); x2['과대예측'] = x2.예측 - x2.cycle_life
sh = x2[x2.구간 == '짧은 덩어리']; V = ['dQ_log_var', 'Tavg', 'chargetime', 'IR', 'Q0']

# ① 짧은 30셀의 입력 값이 Batch 1 범위 밖인 비율
t1 = pd.DataFrame({v: {'Batch 1 범위': f'{x1[v].min():.3g}~{x1[v].max():.3g}', '짧은 30셀 중앙값': sh[v].median(), '범위 밖 셀 수': int(((sh[v] < x1[v].min()) | (sh[v] > x1[v].max())).sum())} for v in V}).T
display(t1.style.format({'짧은 30셀 중앙값': '{:.3g}'}))
print(f'정책: 짧은 30셀 {sh.policy.nunique()}종 중 Batch 1에도 있는 정책 {len(set(sh.policy) & set(x1.policy))}종 {sorted(set(sh.policy) & set(x1.policy))}')

# ② dQ_log_var가 겹치는 구간의 Batch 1 셀과 비교 (같은 feature 값에서 수명이 얼마나 다른가)
lo, hi = sh.dQ_log_var.min(), sh.dQ_log_var.max(); b1 = x1[(x1.dQ_log_var >= lo) & (x1.dQ_log_var <= hi)]
print(f'\ndQ_log_var {lo:.2f}~{hi:.2f} 구간: Batch 1 {len(b1)}셀 / 짧은 30셀')
display(pd.DataFrame({'Batch 1 (같은 구간)': {'셀': len(b1), '실제 수명 평균': b1.cycle_life.mean(), '모델 예측 평균': final.predict(b1[A].values).mean(), 'chargetime 중앙값': b1.chargetime.median()},
                      'Batch 2 짧은 30셀': {'셀': len(sh), '실제 수명 평균': sh.cycle_life.mean(), '모델 예측 평균': sh.예측.mean(), 'chargetime 중앙값': sh.chargetime.median()}}).style.format('{:.1f}'))

# ③ 과대예측(예측−실제)과 같이 움직이는 값 — 셀 단위와 정책 단위(셀 평균)
rows = {}
for v in V:
    pol = sh.groupby('policy')[[v, '과대예측']].mean()
    rows[v] = {'셀 단위 Spearman': spearmanr(sh[v], sh.과대예측)[0], '정책 단위 Spearman': spearmanr(pol[v], pol.과대예측)[0]}
display(pd.DataFrame(rows).T.style.format('{:+.2f}'))
print(f'(셀 단위 n={len(sh)}, 정책 단위 n={sh.policy.nunique()} — 같은 정책 셀은 서로 비슷하므로 정책 단위가 더 보수적)')

,Batch 1 범위,짧은 30셀 중앙값,범위 밖 셀 수
dQ_log_var,-5.01~-3.35,-3.46,11
Tavg,29.8~33.9,32.9,2
chargetime,8.99~13.4,10.2,0
IR,0.0151~0.0199,0.0165,3
Q0,1.06~1.1,1.1,16


정책: 짧은 30셀 9종 중 Batch 1에도 있는 정책 2종 ['4.8C(80%)-4.8C', '6C(60%)-3C']

dQ_log_var -3.71~-3.09 구간: Batch 1 14셀 / 짧은 30셀


,Batch 1 (같은 구간),Batch 2 짧은 30셀
셀,14.0,30.0
실제 수명 평균,658.7,453.0
모델 예측 평균,685.1,617.7
chargetime 중앙값,10.2,10.2


,셀 단위 Spearman,정책 단위 Spearman
dQ_log_var,-0.82,-0.85
Tavg,+0.07,+0.25
chargetime,-0.39,-0.42
IR,+0.12,-0.05
Q0,-0.47,-0.25


(셀 단위 n=30, 정책 단위 n=9 — 같은 정책 셀은 서로 비슷하므로 정책 단위가 더 보수적)


**결과 · 해석 (배치 간 차이)** — 원인은 확정되지 않음
- **입력 값 범위는 큰 문제가 아님**: 짧은 30셀의 온도·충전시간·IR은 대부분 Batch 1 범위 안(범위 밖 Tavg 2셀, chargetime 0셀, IR 3셀). 초기 용량 `Q0`만 16셀이 Batch 1 범위를 벗어나지만 값 차이는 작음(표기상 모두 약 1.1). `chargetime` 평균은 Batch 2에 이상치 사이클이 있어 왜곡되므로 중앙값으로 비교함.
- **정책이 대부분 새로운 것**: 짧은 30셀의 정책 9종 중 Batch 1에도 있는 것은 2종(`4.8C(80%)-4.8C`, `6C(60%)-3C`). 모델은 이 충전 조건들을 거의 본 적이 없음.
- **같은 `dQ_log_var` 구간(−3.71~−3.09)에서 수명이 다름**: Batch 1 14셀 실제 수명 평균 659 vs Batch 2 짧은 30셀 453(약 200사이클 짧음). 이것이 7번에서 본 절편 이동의 실체.
- **과대예측과 같이 움직이는 값**: `dQ_log_var`와 −0.82(셀 단위)/−0.85(정책 단위). `dQ_log_var`가 클수록(모델이 짧게 예측할수록) 과대예측이 작음 → 짧은 30셀 안에서는 모델의 기울기가 실제보다 가파름(셀 안 상관 −0.39로 약함과 일치). 예측이 `dQ_log_var`의 함수라 어느 정도 기계적인 관계이므로 원인 근거로 쓰지 않음. 나머지(Tavg·IR 거의 0, `chargetime` −0.39/−0.42, `Q0` −0.47/−0.25)는 정책 단위 n=9라 약하고 일관되지 않음.
- **결론**: 지금 갖고 있는 값 중에서는 절편 이동을 설명하는 것이 없음. "Batch 1에 없는 충전 조건이 많고, 그 조건의 영향이 `dQ_log_var`에 담기지 않았을 가능성"은 후보일 뿐 확인된 것이 아님. 한계: 짧은 30셀은 정책 9종뿐이라 통계적으로 약함.

## 10. 시사점 · 논문 대비 해석

**비교 (Target: 회귀 MAPE 9.1%)**

| | Train(CV) | Valid(Hold-out) | Test(Batch 2) | Gap(Target-Test) |
|---|---|---|---|---|
| 회귀 MAPE | 7.3% | 13.1% | **31.5%** | +22.4%p |

(분류는 본 분석이 아니라 논문 비교용 참고라 맨 아래 부록에만 정리)

**1. 모델 복잡도의 문제가 아니라 배치 간 일반화의 문제**
- Batch 1 안에서는 논문과 비슷한 수준(CV 7.3%)인데 Batch 2에서만 무너짐. 트리·앙상블로 바꿔도 해결되지 않음(H4 외삽, 앙상블 CV 7.5%).
- Batch 2 자체로 선형을 다시 피팅하면 MAPE 13.6%, `dQ_log_var`–수명 상관도 Batch 1(−0.89)과 Batch 2(−0.90)가 같음 → **feature의 방향은 맞고, 같은 feature 값에서 수명이 150~200사이클 짧은 Batch 2 짧은 셀 때문에 직선이 위로 어긋남(절편 이동)**.
- 10번 섹션에서 확인: 짧은 30셀의 온도·충전시간·IR은 Batch 1 범위 안이고, 정책 9종 중 Batch 1에도 있는 것은 2종뿐. 같은 `dQ_log_var` 구간에서 실제 수명이 약 200사이클 짧음. 그러나 가진 값 중 이 차이를 설명하는 것은 없어 원인(`newstructure`·제조·측정 조건 등)은 여전히 **확인하지 못한 가설**.

**2. 학습 데이터가 평가 데이터의 분포를 덮지 못함**
- Batch 2 중 11셀은 `dQ_log_var`가 학습 범위 밖. Batch 2의 77%(30셀)가 Batch 1 최소 수명(534)보다 짧은 392~514.

**3. 논문과 직접 비교할 때의 한계**
- 논문의 9.1%가 어떤 테스트셋에 대한 어떤 오차 정의인지는 이 노트북에서 직접 확인하지 못함(과제 안내문의 값을 Target으로 사용).
- Valid(hold-out)는 11셀, 3셀이 오차를 좌우함 → Train-Valid Gap(+5.8%p)을 과적합으로 단정하지 않음(선형의 train–valid gap은 +0.5%p).

**4. 다음에 해볼 만한 것 (아직 안 한 것 — 7번의 Batch 2 첫 결과는 그대로 두고, 아래를 하면 수정본으로 따로 표시)**

| 순서 | 할 일 | 무엇을 확인하나 | 방법 | 주의 |
|---|---|---|---|---|
| 1 (일부 완료, 10번) | 배치 간 차이 원인 확인 | Batch 2 짧은 30셀이 같은 `dQ_log_var`에서 수명이 150~200 사이클 짧은 이유 | 새 학습 없이, 이 30셀과 Batch 1의 비슷한 `dQ_log_var` 셀을 정책·온도·충전시간·초기 용량 등으로 비교 | 가설 확인 단계. 원인(제조·측정 조건 등)을 단정하지 않고 "차이가 보인다"까지만 기술 |
| 2 | 학습에 짧은 수명 셀 포함 | 학습 범위(최소 534) 밖이라 생긴 오차인지 | 예: Batch 2 일부를 학습에 넣고 정책 단위로 나눠 검증 | 이 설계의 Batch 2 점수는 낙관적 → 7번 결과와 **별도 표**로 표시. Batch 3 재평가는 팀 합의 후 |
| 3 | (참고) 초기 5사이클 feature로 분류 직접 학습 | 부록의 논문 분류 4.9%와 같은 조건에서의 비교 — 본 분석(회귀) 범위 밖 | 초기 5사이클에서 feature를 새로 설계해 550 기준 분류 | Batch 1에 550 미만이 1셀뿐이라 단수명 학습 데이터 부족(2번과 같이 풀어야 의미 있음) |

- 순서 근거: 1번은 재학습이 필요 없고 핵심 발견(절편 이동)의 원인을 직접 좁힘. 2번은 가장 직접적인 해결책이지만 Batch 2의 "처음 보는 데이터" 성격을 잃음. 3번은 논문 조건에는 가깝지만 1·2번이 풀리지 않으면 같은 한계(단수명 셀 부족)에 부딪힘.
- 이번 과제 범위: 1번까지 해도 충분. 2·3번은 시간이 되면 하고, 안 하면 한계로 기술.

## 11. 팀원 결과와 비교

같은 데이터(Batch 1 학습 → Batch 2 테스트)·같은 모델·feature에서 **검증 분할 세부만 다르게** 독립적으로 수행한 결과. 팀원 수치는 팀원이 보내온 표를 그대로 옮김(내가 재현하지 않음).

| 구분 | 내 결과 | 팀원 결과 |
|---|---|---|
| 분할 | hold-out 정책 5종(11셀), 정책 단위 5-fold CV 5회 반복 | hold-out 정책 5종(11셀, 2개 정책 다름), `GroupKFold(5)` |
| 모델 · Feature | `dQ_log_var` 1개 + 선형 | Linear Regression + `deltaQ_logvar` 1개 |
| Train (CV) MAPE | 7.3% | 7.60% ± 1.57% |
| Valid (hold-out) MAPE | 13.1% | 11.63% |
| **Test (Batch 2) MAPE** | **31.5%** | **31.53%** |

**해석**
- **Test는 사실상 같음(31.5% vs 31.53%).** Test는 Batch 1 전체로 학습한 모델을 Batch 2에 적용하므로 검증 분할에 영향받지 않음. 독립적으로 구현한 두 코드가 같은 값을 내므로 **계산 오류가 아니라 데이터·모델 구조에서 나온 결과**라는 근거가 됨(모델·feature가 같다는 전제).
- **Train(CV)은 팀원 ±1.57% 안에서 일치(7.3% vs 7.60%).** 분할 방식이 달라도 Batch 1 안의 성능은 7%대로 같음. `±`가 무엇의 표준편차인지는 확인하지 못함(fold 표준편차로 추정).
- **Valid(hold-out)는 1.5%p 차이(13.1% vs 11.63%).** hold-out 정책 5개 중 2개가 달라서 생긴 차이일 수 있으나, 같은 manifest로 확인한 것은 아님(가설). 어느 쪽이든 11셀 hold-out이 어떤 정책이 들어가느냐에 따라 달라질 만큼 불안정하다는 점은 일치(내 쪽에서도 3셀이 오차를 좌우).
- **세 구간 모두 같은 방향**: Batch 1 안(7~8%, 11~13%)에서는 괜찮고 Batch 2에서만 31.5% → 분할을 달리해도 **배치 간 일반화 문제**라는 결론이 유지됨.

## 부록. 논문 성능 비교용 분류 (과제 본 분석 아님)

안내문은 회귀 / 분류 중 **한 가지만** 선택하도록 되어 있고, 본 분석은 **회귀**를 선택함. 아래는 논문의 분류 성능(1−Acc 4.9%)과 대략 견주어 보기 위한 **참고 틀**이며, 모델 선택·최종 결론·위 Reporting format에는 사용하지 않음.

과제 기준: `label = (cycle_life >= 550)`(1 = 장수명), 지표는 1−Accuracy와 F1. **새로 학습·예측하지 않고 7번의 Batch 2 예측값을 재사용**(모델·feature 변경 없음). 논문 분류는 **초기 5사이클**로 직접 분류하지만 여기서는 **사이클 10·100 기반 회귀 예측을 이진화**한 것이라 입력이 달라 같은 조건의 비교가 아님.

In [14]:
TH = 550
def cls_report(true, pred):
    t, p = true >= TH, pred >= TH
    return {'셀': len(t), '장수명(실제)': t.sum(), '1−Accuracy': (t != p).mean(), '단→장 오분류': (~t & p).sum(), '장→단 오분류': (t & ~p).sum()}

# Batch 1 CV: 반복 평균 OOF 예측이 아니라 반복별로 오차를 계산해 평균
oof = runs[('Linear', 'A')]['oof']
b1 = [cls_report(y[tr_all], oof[r, tr_all]) for r in range(N_REP)]
rows = {'Batch 1 CV (35셀, 반복 평균)': {k: np.mean([b[k] for b in b1]) for k in b1[0]},
        'Batch 2 전체': cls_report(c2.cycle_life.values, c2.예측.values)}
for k, g in c2.groupby('구간'): rows[f'Batch 2 {k}'] = cls_report(g.cycle_life.values, g.예측.values)
display(pd.DataFrame(rows).T.style.format({'셀': '{:.0f}', '장수명(실제)': '{:.0f}', '1−Accuracy': '{:.1%}', '단→장 오분류': '{:.1f}', '장→단 오분류': '{:.1f}'}))
print('Batch 1 학습 수명 최소', y.min(), '→ 550 미만 셀', (y < TH).sum(), '개 (분류 경계 근처 학습 데이터가 거의 없음)')
print('Batch 2 예측 중 최솟값', c2.예측.min().round(0), '/ 550 미만으로 예측한 셀', (c2.예측 < TH).sum(), '개')

,셀,장수명(실제),1−Accuracy,단→장 오분류,장→단 오분류
"Batch 1 CV (35셀, 반복 평균)",35,34,4.0%,1.0,0.4
Batch 2 전체,39,9,59.0%,23.0,0.0
Batch 2 newstructure(9),9,9,0.0%,0.0,0.0
Batch 2 짧은 덩어리,30,0,76.7%,23.0,0.0


Batch 1 학습 수명 최소 534.0 → 550 미만 셀 1 개 (분류 경계 근처 학습 데이터가 거의 없음)
Batch 2 예측 중 최솟값 482.0 / 550 미만으로 예측한 셀 7 개


**설명 (분류 관점)**
- 새로 학습하지 않고 7번의 Batch 2 예측값에 `cycle_life >= 550` 기준을 적용(장수명=1). 지표는 1−Accuracy.
- 읽는 법: Batch 1 CV 4.0%는 논문 4.9%와 비슷해 보이지만, 35셀 중 34셀이 장수명이라 전부 "장수명"이라고 찍어도 2.9%. 의미 있는 것은 Batch 2의 59.0%이며, 오분류 23건이 전부 짧은 덩어리를 장수명으로 잘못 본 경우.
- 주의: 논문은 초기 5사이클로 직접 분류, 여기는 사이클 10·100 기반 회귀 예측을 이진화한 것이라 같은 조건이 아님.

In [15]:
from sklearn.metrics import accuracy_score, f1_score
def clf(t, p):   # 장수명(>=550)을 양성으로. macro F1은 두 클래스 평균(한 클래스만 있으면 F1이 퇴화하므로 같이 표기)
    t, p = t >= TH, p >= TH
    return {'Acc': accuracy_score(t, p), 'F1(장수명)': f1_score(t, p, zero_division=0), 'macro F1': f1_score(t, p, average='macro', zero_division=0)}
oof = runs[('Linear', 'A')]['oof']
rep = pd.DataFrame({'Train (Batch 1 CV)': pd.DataFrame([clf(y[tr_all], oof[r, tr_all]) for r in range(N_REP)]).mean().to_dict(),
                    'Valid (Batch 1 Hold-out)': clf(y[ho], p_ho), 'Test (Batch 2)': clf(c2.cycle_life.values, c2.예측.values)}).T
rep['1−Acc'] = 1 - rep.Acc
display(rep.style.format('{:.1%}'))
print(f'Gap (Target-Test): 59.0% − 4.9% = {(rep.loc["Test (Batch 2)", "1−Acc"] - 0.049):+.1%}p (Target: 논문 분류 1−Acc 4.9%)')
print(f'hold-out 11셀 중 장수명 {(y[ho] >= TH).sum()}셀 → 단수명이 없어 F1·macro F1이 퇴화(해석 불가)')

,Acc,F1(장수명),macro F1,1−Acc
Train (Batch 1 CV),96.0%,98.0%,49.0%,4.0%
Valid (Batch 1 Hold-out),100.0%,100.0%,100.0%,0.0%
Test (Batch 2),41.0%,43.9%,40.9%,59.0%


Gap (Target-Test): 59.0% − 4.9% = +54.1%p (Target: 논문 분류 1−Acc 4.9%)
hold-out 11셀 중 장수명 11셀 → 단수명이 없어 F1·macro F1이 퇴화(해석 불가)


**설명 (분류 Accuracy·F1 표)**
- 위 분류 오차를 Train/Valid/Test로 나눠 Accuracy·F1로 정리. 새 예측 없음.
- 읽는 법: Train·Valid는 단수명 셀이 거의 없어(CV 35셀 중 34셀, hold-out 11셀 전부 장수명) 의미가 약함. 의미 있는 값은 Batch 2뿐이며, 회귀에서 본 학습 범위 문제와 같은 현상(짧은 30셀 중 23셀을 장수명으로 오분류).